In [ ]:
import pandas as pd
import re
import os

df = pd.read_csv("Artworks.csv", low_memory=False)
print("Original rows:", len(df))

if "Date" not in df.columns:
    raise ValueError("The original CSV needs a column named Date.")

print(df["Date"].head())


In [ ]:
def find_year(date):
    if pd.isna(date):
        return None, None, "missing"

    text = str(date).strip().lower()

    if text == "" or text in ["unknown", "n.d.", "n.d", "undated"]:
        return None, None, "missing"

    if "before" in text or "after" in text:
        return None, None, "unclear"

    decade = re.search(r"(?<!\d)(\d{3,4})s(?!\w)", text)

    short_range = re.search(r"(\d{4})\s*[-–/]\s*(\d{2})(?!\d)", text)

    years = re.findall(r"(?<!\d)\d{3,4}(?!\d)", text)

    if decade:
        lower = int(decade.group(1))
        upper = lower + 9
        date_type = "decade"
    elif short_range:
        lower = int(short_range.group(1))
        upper = int(str(lower)[:2] + short_range.group(2))
        if upper < lower:
            upper = upper + 100
        date_type = "range"
    elif len(years) >= 2:
        lower = int(years[0])
        upper = int(years[1])
        date_type = "range"
    elif len(years) == 1:
        lower = int(years[0])
        upper = lower
        date_type = "exact"
    else:
        return None, None, "unclear"

    simple_text = text.replace(".", "")
    if "bc" in simple_text:
        lower, upper = -max(lower, upper), -min(lower, upper)
        date_type = "BC"

    if lower > upper:
        lower, upper = upper, lower

    if date_type == "exact" and ("circa" in text or "c." in text or "ca." in text):
        date_type = "circa"

    return lower, upper, date_type

print(find_year("1968"))
print(find_year("1934-36"))
print(find_year("1960s"))


In [ ]:
lower_years = []
upper_years = []
middle_years = []
date_types = []

for date in df["Date"]:
    lower, upper, date_type = find_year(date)
    lower_years.append(lower)
    upper_years.append(upper)
    date_types.append(date_type)

    if lower is None:
        middle_years.append(None)
    else:
        middle_years.append((lower + upper) / 2)

df["year_lower"] = lower_years
df["year_upper"] = upper_years
df["year_mid"] = middle_years
df["date_type"] = date_types

clean = df.dropna(subset=["year_mid"]).copy()

output_file = "Artworks_dates_analysis.csv"
if os.path.exists(output_file):
    output_file = "Artworks_dates_rebuilt.csv"

clean.to_csv(output_file, index=False)

print("Cleaned rows:", len(clean))
print("Full-data mean:", round(clean["year_mid"].mean(), 6))
print("Full-data SD:", round(clean["year_mid"].std(ddof=0), 6))
print("Years before 1800:", (clean["year_mid"] < 1800).sum())
print("Saved:", output_file)

if len(clean) != 155118:
    print("Check: row count differs from the slide's 155118 rows.")
